# Explainability Methods Verification & Testing

**Objective:** Systematically test all four explainability methods (Grad-CAM, Attention Maps, Integrated Gradients, SHAP) on top-performing models (DenseNet121 + Swin Tiny) to verify they work end-to-end and compute quantitative metrics for comparison.

**Timeline:** This week (immediate)
**Focus:** Manual verification with actual execution
**Coverage:** Top 2 performing architectures (DenseNet121 perclass, Swin Tiny perclass)

## Test Plan

| # | Method | DenseNet121 | Swin Tiny | Status | Notes |
|---|--------|------------|-----------|--------|-------|
| 1 | Grad-CAM | 3 samples | 3 samples | ⏳ TODO | Fast (~10 sec/sample) |
| 2 | Attention Maps | N/A (CNN) | 3 samples | ⏳ TODO | Transformer-only |
| 3 | Integrated Gradients | 3 samples | 3 samples | ⏳ TODO | Medium (~10 sec/sample) |
| 4 | SHAP + SanityCheck | 1 sample | 1 sample | ⏳ TODO | Slow (10-30 min/sample) |

**Total expected runtime:** 1-2 hours (excluding SHAP which is 20-40 min)

---

## Setup: Load Environment


In [12]:
import os
import sys
import subprocess
import json
import time
from datetime import datetime
from pathlib import Path

# Paths
WORK_DIR = "/Users/bennistieger/Desktop/Thesis"
os.chdir(WORK_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"Python: {sys.version}")

# Check that required scripts exist
SCRIPTS = {
    "gradcam": "scripts/gradcam.py",
    "attention_maps": "scripts/attention_maps.py",
    "integrated_gradients": "scripts/integrated_gradients.py",
    "shap_analysis": "scripts/shap_analysis.py",
    "compute_metrics": "scripts/compute_explainability_metrics.py",
}

for name, script_path in SCRIPTS.items():
    exists = os.path.exists(script_path)
    status = "✅" if exists else "❌"
    print(f"{status} {name:20s} {script_path}")

# Create output directories
for subdir in ["gradcam", "attention_maps", "integrated_gradients", "shap", "shap_sanity_checks"]:
    os.makedirs(f"outputs/{subdir}", exist_ok=True)
    print(f"✅ Created outputs/{subdir}")


Working directory: /Users/bennistieger/Desktop/Thesis
Python: 3.13.3 (v3.13.3:6280bb54784, Apr  8 2025, 10:47:54) [Clang 15.0.0 (clang-1500.3.9.4)]
✅ gradcam              scripts/gradcam.py
✅ attention_maps       scripts/attention_maps.py
✅ integrated_gradients scripts/integrated_gradients.py
✅ shap_analysis        scripts/shap_analysis.py
✅ compute_metrics      scripts/compute_explainability_metrics.py
✅ Created outputs/gradcam
✅ Created outputs/attention_maps
✅ Created outputs/integrated_gradients
✅ Created outputs/shap
✅ Created outputs/shap_sanity_checks


In [13]:
# Helper functions
import subprocess
from typing import Dict, List

# Logging structure to track all tests
test_log = {
    "start_time": datetime.now().isoformat(),
    "tests": []
}

def run_explainability_test(method: str, model: str, indices: List[int], sanity_check: bool = False, n_background: int = 30) -> Dict:
    """Run an explainability method on a model with given test samples."""
    
    # Map model name to config and checkpoint
    model_config_map = {
        "densenet121": {
            "config": "configs/densenet121_perclass_longer.yaml",
            "checkpoint": "checkpoints/densenet121/run011_densenet121_perclass_longer_ep07_val0.7931_test0.8145.pt"
        },
        "swin_tiny": {
            "config": "configs/swin_tiny_perclass.yaml",
            "checkpoint": "checkpoints/swin_tiny/run008_swin_tiny_perclass_ep09_val0.7937_test0.8129.pt"
        }
    }
    
    if model not in model_config_map:
        print(f"❌ Unknown model: {model}")
        return {"success": False, "error": f"Unknown model {model}"}
    
    config = model_config_map[model]["config"]
    checkpoint = model_config_map[model]["checkpoint"]
    
    # Build command
    if method == "gradcam":
        cmd_template = ["uv", "run", "python", "-m", "scripts.gradcam", "--config", config, "--checkpoint", checkpoint, "--index", "{index}", "--output", "outputs/gradcam"]
    elif method == "attention_maps":
        cmd_template = ["uv", "run", "python", "-m", "scripts.attention_maps", "--config", config, "--checkpoint", checkpoint, "--index", "{index}", "--output", "outputs/attention_maps"]
    elif method == "integrated_gradients":
        cmd_template = ["uv", "run", "python", "-m", "scripts.integrated_gradients", "--config", config, "--checkpoint", checkpoint, "--index", "{index}", "--n-steps", "50", "--output", "outputs/integrated_gradients"]
    elif method == "shap_analysis":
        cmd_template = ["uv", "run", "python", "-m", "scripts.shap_analysis", "--config", config, "--checkpoint", checkpoint, "--index", "{index}", "--n-background", str(n_background), "--output", "outputs/shap"]
        if sanity_check:
            cmd_template.append("--run-sanity-check")
    else:
        print(f"❌ Unknown method: {method}")
        return {"success": False, "error": f"Unknown method {method}"}
    
    # Run for each index
    results = {"method": method, "model": model, "samples": []}
    
    for idx in indices:
        cmd = [arg.replace("{index}", str(idx)) for arg in cmd_template]
        print(f"\n{'='*70}")
        print(f" {method.upper()} on {model.upper()} [sample idx={idx}]")
        print(f"{'='*70}")
        print(f"$ {' '.join(cmd)}")
        print()
        
        try:
            start_time = time.time()
            result = subprocess.run(cmd, capture_output=False, text=True, check=False)
            elapsed = time.time() - start_time
            
            success = result.returncode == 0
            results["samples"].append({
                "index": idx,
                "success": success,
                "elapsed_sec": elapsed,
                "returncode": result.returncode
            })
            
            status = "✅" if success else "❌"
            print(f"{status} Completed in {elapsed:.1f} seconds (return code: {result.returncode})")
            
        except Exception as e:
            print(f"❌ Exception: {e}")
            results["samples"].append({
                "index": idx,
                "success": False,
                "error": str(e)
            })
    
    return results

# Record test start
print("🚀 Starting Explainability Verification Tests")
print(f"   Start time: {test_log['start_time']}")


🚀 Starting Explainability Verification Tests
   Start time: 2026-03-24T18:37:06.611086


---

## Phase 1: Test Grad-CAM (Fast, ~30 seconds total)

Grad-CAM is the fastest method. Testing on 3 samples per model.


In [14]:
# Test Grad-CAM on DenseNet121
print("\n" + "="*70)
print(" TESTING: Grad-CAM on DenseNet121 (3 samples)")
print("="*70)
result_gc_dn = run_explainability_test("gradcam", "densenet121", [0, 1, 2])
test_log["tests"].append(result_gc_dn)

# Test Grad-CAM on Swin Tiny
print("\n" + "="*70)
print(" TESTING: Grad-CAM on Swin Tiny (3 samples)")
print("="*70)
result_gc_sw = run_explainability_test("gradcam", "swin_tiny", [0, 1, 2])
test_log["tests"].append(result_gc_sw)

# Summary
gc_success = sum(1 for s in result_gc_dn["samples"] if s["success"]) + sum(1 for s in result_gc_sw["samples"] if s["success"])
print(f"\n✨ Grad-CAM Summary: {gc_success}/6 samples successful")



 TESTING: Grad-CAM on DenseNet121 (3 samples)

 GRADCAM on DENSENET121 [sample idx=0]
$ uv run python -m scripts.gradcam --config configs/densenet121_perclass_longer.yaml --checkpoint checkpoints/densenet121/run011_densenet121_perclass_longer_ep07_val0.7931_test0.8145.pt --index 0 --output outputs/gradcam

Test image [0]: CheXpert-v1.0-small/train/patient00082/study1/view1_frontal.jpg
Loaded densenet121 from epoch 7
Saved: outputs/gradcam/patient00082_study1_view1_frontal_densenet121_gradcam.png
✅ Completed in 12.1 seconds (return code: 0)

 GRADCAM on DENSENET121 [sample idx=1]
$ uv run python -m scripts.gradcam --config configs/densenet121_perclass_longer.yaml --checkpoint checkpoints/densenet121/run011_densenet121_perclass_longer_ep07_val0.7931_test0.8145.pt --index 1 --output outputs/gradcam

Test image [1]: CheXpert-v1.0-small/train/patient35759/study9/view1_frontal.jpg
Loaded densenet121 from epoch 7
Saved: outputs/gradcam/patient35759_study9_view1_frontal_densenet121_gradcam.pn

---

## Phase 2: Test Attention Maps (Transformers only, ~20 seconds)

Attention maps only work on Swin Tiny and ViT Base. Testing on Swin Tiny only (ViT not in top performers).


In [15]:
# Test Attention Maps on Swin Tiny
print("\n" + "="*70)
print(" TESTING: Attention Maps on Swin Tiny (3 samples)")
print("="*70)
result_att_sw = run_explainability_test("attention_maps", "swin_tiny", [0, 1, 2])
test_log["tests"].append(result_att_sw)

att_success = sum(1 for s in result_att_sw["samples"] if s["success"])
print(f"\n✨ Attention Maps Summary: {att_success}/3 samples successful")



 TESTING: Attention Maps on Swin Tiny (3 samples)

 ATTENTION_MAPS on SWIN_TINY [sample idx=0]
$ uv run python -m scripts.attention_maps --config configs/swin_tiny_perclass.yaml --checkpoint checkpoints/swin_tiny/run008_swin_tiny_perclass_ep09_val0.7937_test0.8129.pt --index 0 --output outputs/attention_maps

Test image [0]: CheXpert-v1.0-small/train/patient00082/study1/view1_frontal.jpg
Loaded swin_tiny from epoch 9
Extracting attention map for swin_tiny...
Saved: outputs/attention_maps/patient00082_study1_view1_frontal_swin_tiny_attention.png
✅ Completed in 7.1 seconds (return code: 0)

 ATTENTION_MAPS on SWIN_TINY [sample idx=1]
$ uv run python -m scripts.attention_maps --config configs/swin_tiny_perclass.yaml --checkpoint checkpoints/swin_tiny/run008_swin_tiny_perclass_ep09_val0.7937_test0.8129.pt --index 1 --output outputs/attention_maps

Test image [1]: CheXpert-v1.0-small/train/patient35759/study9/view1_frontal.jpg
Loaded swin_tiny from epoch 9
Extracting attention map for swin

---

## Phase 3: Test Integrated Gradients (Medium speed, ~1 minute total)

Integrated Gradients work on all architectures. Uses 50 steps for speed (standard: 100).


In [16]:
# Test Integrated Gradients on DenseNet121
print("\n" + "="*70)
print(" TESTING: Integrated Gradients on DenseNet121 (3 samples)")
print("="*70)
result_ig_dn = run_explainability_test("integrated_gradients", "densenet121", [0, 1, 2])
test_log["tests"].append(result_ig_dn)

# Test Integrated Gradients on Swin Tiny
print("\n" + "="*70)
print(" TESTING: Integrated Gradients on Swin Tiny (3 samples)")
print("="*70)
result_ig_sw = run_explainability_test("integrated_gradients", "swin_tiny", [0, 1, 2])
test_log["tests"].append(result_ig_sw)

ig_success = sum(1 for s in result_ig_dn["samples"] if s["success"]) + sum(1 for s in result_ig_sw["samples"] if s["success"])
print(f"\n✨ Integrated Gradients Summary: {ig_success}/6 samples successful")



 TESTING: Integrated Gradients on DenseNet121 (3 samples)

 INTEGRATED_GRADIENTS on DENSENET121 [sample idx=0]
$ uv run python -m scripts.integrated_gradients --config configs/densenet121_perclass_longer.yaml --checkpoint checkpoints/densenet121/run011_densenet121_perclass_longer_ep07_val0.7931_test0.8145.pt --index 0 --n-steps 50 --output outputs/integrated_gradients

Test image [0]: CheXpert-v1.0-small/train/patient00082/study1/view1_frontal.jpg
Note: Using CPU instead of MPS for Captum compatibility
Loaded densenet121 from epoch 7
Computing Integrated Gradients with 50 steps...
  [1/5] Cardiomegaly: done
  [2/5] Edema: done
  [3/5] Consolidation: done
  [4/5] Atelectasis: done
  [5/5] Pleural Effusion: done
Saved: outputs/integrated_gradients/patient00082_study1_view1_frontal_densenet121_ig.png
✅ Completed in 37.2 seconds (return code: 0)

 INTEGRATED_GRADIENTS on DENSENET121 [sample idx=1]
$ uv run python -m scripts.integrated_gradients --config configs/densenet121_perclass_longer

---

## Phase 4: Test SHAP with Model Randomization Sanity Check (SLOW, ~30-40 minutes)

🐢 **WARNING: This phase is SLOW** 
- DenseNet121: ~15-20 min per sample (uses CPU)
- Swin Tiny: ~20-40 min per sample (uses CPU, Transformer overhead)
- Testing only **1 sample per model** with `n-background=30` for speed

This is the **critical untested method**. The sanity check validates that SHAP values depend on the model weights.


In [17]:
# Test SHAP with Sanity Check on DenseNet121 (n-background=30 for speed)
print("\n" + "="*70)
print(" TESTING: SHAP + Sanity Check on DenseNet121 (1 sample, n-bg=30)")
print(" ⏳ This will take ~15 minutes...")
print("="*70)
result_shap_dn = run_explainability_test("shap_analysis", "densenet121", [0], sanity_check=True, n_background=30)
test_log["tests"].append(result_shap_dn)

# Test SHAP with Sanity Check on Swin Tiny (n-background=30 for speed)
print("\n" + "="*70)
print(" TESTING: SHAP + Sanity Check on Swin Tiny (1 sample, n-bg=30)")
print(" ⏳ This will take ~20-30 minutes...")
print("="*70)
result_shap_sw = run_explainability_test("shap_analysis", "swin_tiny", [0], sanity_check=True, n_background=30)
test_log["tests"].append(result_shap_sw)

shap_success = sum(1 for s in result_shap_dn["samples"] if s["success"]) + sum(1 for s in result_shap_sw["samples"] if s["success"])
print(f"\n✨ SHAP + Sanity Check Summary: {shap_success}/2 samples successful")
print("\n🔍 Check outputs/shap/ for:")
print("   - {patient}_{arch}_shap.png (main visualization)")
print("   - {patient}_{arch}_shap_sanitycheck.png (model randomization test)")



 TESTING: SHAP + Sanity Check on DenseNet121 (1 sample, n-bg=30)
 ⏳ This will take ~15 minutes...

 SHAP_ANALYSIS on DENSENET121 [sample idx=0]
$ uv run python -m scripts.shap_analysis --config configs/densenet121_perclass_longer.yaml --checkpoint checkpoints/densenet121/run011_densenet121_perclass_longer_ep07_val0.7931_test0.8145.pt --index 0 --n-background 30 --output outputs/shap --run-sanity-check

Test image [0]: CheXpert-v1.0-small/train/patient00082/study1/view1_frontal.jpg

⚠️ SHAP is incompatible with MPS (Apple Silicon). Falling back to CPU.
   This will be slower (10-30 min per image), but necessary for stability.
Loaded densenet121 from epoch 7
Loading 30 background samples from training set...
Background shape: torch.Size([30, 3, 224, 224])
Computing SHAP values for 5 labels...
  [1/5] Cardiomegaly: done
  [2/5] Edema: done
  [3/5] Consolidation: done
  [4/5] Atelectasis: done
  [5/5] Pleural Effusion: done
Saved: outputs/shap/patient00082_study1_view1_frontal_densenet121

---

## Phase 5: Compute Quantitative Metrics

Run the metrics script to compute:
- **Heatmap IoU:** How much agreement between Grad-CAM and Integrated Gradients?
- **Pixel Correlation:** Pearson r between heatmap values
- **Attention Entropy:** How localized are Swin Tiny's attention patterns?
- **Structural Similarity (SSIM):** Overall visual agreement between methods


In [18]:
# Compute metrics for all generated heatmaps
import os

print("\n" + "="*70)
print(" Computing Quantitative Metrics")
print("="*70)

metric_cmd = ["uv", "run", "python", "-m", "scripts.compute_explainability_metrics", 
              "--output-dir", "outputs/", 
              "--methods", "gradcam", "integrated_gradients", "attention_maps", "shap"]

print(f"$ {' '.join(metric_cmd)}\n")

try:
    result = subprocess.run(metric_cmd, capture_output=False, text=True, check=False)
    if result.returncode == 0:
        print("\n✅ Metrics computed successfully!")
        if os.path.exists("outputs/explainability_metrics.csv"):
            print("   📊 outputs/explainability_metrics.csv")
        if os.path.exists("outputs/metrics_summary.png"):
            print("   📊 outputs/metrics_summary.png")
    else:
        print(f"\n⚠️ Metrics computation had issues (return code: {result.returncode})")
except Exception as e:
    print(f"❌ Error running metrics: {e}")



 Computing Quantitative Metrics
$ uv run python -m scripts.compute_explainability_metrics --output-dir outputs/ --methods gradcam integrated_gradients attention_maps shap

📁 Scanning: outputs/
🔍 Methods: gradcam, integrated_gradients, attention_maps, shap

📊 Found 4 unique samples
  [1/4] Computing metrics for patient00082...
  [2/4] Computing metrics for patient35759...
  [3/4] Computing metrics for patient38491...
  [4/4] Computing metrics for view1...

✅ Saved metrics CSV: outputs/explainability_metrics.csv

METRICS SUMMARY
       heatmap_iou_mean  heatmap_iou_std  heatmap_iou_count  ssim_mean  ssim_std  attention_entropy_mean  attention_entropy_std  attention_entropy_count  method_correlation_mean  method_correlation_std  shap_sanity_check_count
count          4.000000         4.000000                4.0   4.000000  4.000000                4.000000               4.000000                 4.000000                 4.000000                4.000000                      4.0
mean        

---

## Summary & Results


In [19]:
# Generate Summary Report
import pandas as pd
import glob

test_log["end_time"] = datetime.now().isoformat()

print("\n\n")
print("╔" + "="*68 + "╗")
print("║" + " "*15 + "EXPLAINABILITY VERIFICATION REPORT" + " "*19 + "║")
print("╚" + "="*68 + "╝")

# Test results table
print("\n📋 TEST RESULTS BY METHOD:")
print("-" * 70)

for test in test_log["tests"]:
    method = test["method"]
    model = test["model"]
    samples = test["samples"]
    successful = sum(1 for s in samples if s["success"])
    total = len(samples)
    avg_time = sum(s.get("elapsed_sec", 0) for s in samples if s["success"]) / max(successful, 1) if successful > 0 else 0
    
    status = "✅" if successful == total else ("⚠️" if successful > 0 else "❌")
    print(f"{status} {method:20s} + {model:12s}: {successful:2d}/{total:2d} samples | avg: {avg_time:6.1f}s")

# Output files summary
print("\n📁 OUTPUT FILES GENERATED:")
print("-" * 70)

output_dirs = {
    "Grad-CAM": "outputs/gradcam",
    "Attention Maps": "outputs/attention_maps",
    "Integrated Gradients": "outputs/integrated_gradients",
    "SHAP": "outputs/shap",
    "SHAP Sanity Checks": "outputs/shap_sanity_checks"
}

for name, path in output_dirs.items():
    if os.path.isdir(path):
        files = os.listdir(path)
        count = len(files)
        print(f"✅ {name:20s}: {count:3d} files")
        if count > 0 and count <= 3:
            for f in sorted(files)[:5]:
                print(f"   - {f}")

# Check for metrics
if os.path.exists("outputs/explainability_metrics.csv"):
    df = pd.read_csv("outputs/explainability_metrics.csv")
    print(f"\n📊 Metrics CSV: {len(df)} samples analyzed")
    print(f"   - Available metrics: {', '.join([c for c in df.columns if c != 'timestamp' and c != 'sample_id'])}")

print("\n" + "="*70)
print("✅ Verification complete!")
print("="*70)

print("\n📝 NEXT STEPS:")
print("   1. Review generated PNG files in outputs/")
print("   2. Check SHAP sanity check plots for model dependence")
print("   3. Examine metrics in outputs/explainability_metrics.csv")
print("   4. Create comparison visualizations for thesis")





╔====================================================================╗
║               EXPLAINABILITY VERIFICATION REPORT                   ║
╚====================================================================╝

📋 TEST RESULTS BY METHOD:
----------------------------------------------------------------------
✅ gradcam              + densenet121 :  3/ 3 samples | avg:   10.4s
✅ gradcam              + swin_tiny   :  3/ 3 samples | avg:    7.3s
✅ attention_maps       + swin_tiny   :  3/ 3 samples | avg:    7.1s
✅ integrated_gradients + densenet121 :  3/ 3 samples | avg:   37.4s
✅ integrated_gradients + swin_tiny   :  3/ 3 samples | avg:   32.8s
✅ shap_analysis        + densenet121 :  1/ 1 samples | avg: 1001.1s
✅ shap_analysis        + swin_tiny   :  1/ 1 samples | avg:  708.2s

📁 OUTPUT FILES GENERATED:
----------------------------------------------------------------------
✅ Grad-CAM            :   7 files
✅ Attention Maps      :   4 files
✅ Integrated Gradients:   6 files
✅ SHAP    